In [ ]:
# ====================== STEP 1: INSTALL ALL DEPENDENCIES ======================
print(" Installing system packages...")
!apt-get update -qq
!apt-get install -y zstd -qq


print(" Installing Python packages...")
!pip install -q pymupdf sentence-transformers faiss-cpu flask flask-cors pyngrok

print(" Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("\n All installations completed!")

In [ ]:

import subprocess
import time

print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

print("Pulling model (llama3.2:3b - fast for Colab)...")
subprocess.run(["ollama", "pull", "llama3.2:3b"])

print("\n Ollama + Model is ready!")


In [ ]:

import fitz
import faiss
import numpy as np
import re
import subprocess
import threading
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok

# Configuration
PDF_PATH     = None
CHUNK_SIZE   = 550
CHUNK_OVERLAP = 100
TOP_K        = 5

# Global state
embedder = None
index    = None
chunks   = None

print(" Libraries imported and config ready")

In [ ]:

def parse_and_chunk(pdf_path):
    print("Parsing PDF...")
    doc = fitz.open(pdf_path)
    chunks = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        words = page.get_text("words")
        if not words:
            continue

        from collections import defaultdict
        lines = defaultdict(list)
        for w in words:
            y_key = round(w[1])
            lines[y_key].append(w)

        visual_lines = []
        prev_y1 = None
        for y in sorted(lines.keys()):
            line_words = sorted(lines[y], key=lambda w: w[0])
            line_text = " ".join(w[4] for w in line_words).strip()
            if not line_text:
                continue
            x0 = min(w[0] for w in line_words)
            y0 = min(w[1] for w in line_words)
            x1 = max(w[2] for w in line_words)
            y1 = max(w[3] for w in line_words)
            gap = (y0 - prev_y1) if prev_y1 is not None else 0
            visual_lines.append({
                "text": line_text,
                "page": page_num,
                "bbox": (x0, y0, x1, y1),
                "y1": y1,
                "gap": gap
            })
            prev_y1 = y1

        current_text = ""
        current_spans = []

        for vline in visual_lines:
            if vline["gap"] > 20 and current_text.strip():
                if len(current_text.strip()) > 80:
                    chunks.append({
                        "id": len(chunks),
                        "text": current_text.strip(),
                        "spans": current_spans.copy(),
                        "pages": [page_num]
                    })
                current_text = ""
                current_spans = []

            current_text += vline["text"] + " "
            current_spans.append(vline)

        if len(current_text.strip()) > 80:
            chunks.append({
                "id": len(chunks),
                "text": current_text.strip(),
                "spans": current_spans.copy(),
                "pages": [page_num]
            })

    print(f" Created {len(chunks)} chunks")
    return chunks


def reset_rag_system():
    """Call this function whenever you change the PDF"""
    global index, chunks, PDF_PATH
    # NOTE: embedder is kept loaded — no need to reload model every time
    index    = None
    chunks   = None
    PDF_PATH = None
    print(" RAG System Reset - Ready for new PDF")


def rag_query(question):
    print(f"\nQuery: {question}")

    TOP_K = 10
    q_vec = embedder.encode([question], normalize_embeddings=True)
    scores, indices = index.search(q_vec, TOP_K)

    question_lower = question.lower().strip()

    # Expand common abbreviations
    question_lower = question_lower.replace(" ai", " artificial intelligence")
    question_lower = question_lower.replace("ai ", "artificial intelligence ")

    stop_words = {"what", "is", "are", "who", "how", "why", "when", "where",
                  "define", "explain", "tell", "me", "about", "the", "a", "an",
                  "does", "do", "was", "were", "give", "list", "describe", "of",
                  "in", "on", "at", "to", "for", "with", "and", "or"}
    words = [w.strip("?.,!") for w in question_lower.split()]
    keywords = [w for w in words if w not in stop_words and len(w) >= 2]

    phrases = []
    if len(keywords) >= 2:
        phrases.append(" ".join(keywords))
        for i in range(len(keywords) - 1):
            phrases.append(keywords[i] + " " + keywords[i+1])
    phrases.extend(keywords)

    print(f"Keywords: {keywords}")
    print(f"Phrases:  {phrases}")

    top_chunks = []
    for i, score in zip(indices[0], scores[0]):
        chunk_text_lower = chunks[i]["text"].lower()
        phrase_match = any(phrase in chunk_text_lower for phrase in phrases)
        if score > 0.50 and phrase_match:
            keyword_hits = sum(1 for kw in keywords if kw in chunk_text_lower)
            combined = score + (keyword_hits * 0.1)
            top_chunks.append((combined, chunks[i]))

    top_chunks.sort(key=lambda x: x[0], reverse=True)
    top_chunks = [chunk for _, chunk in top_chunks[:1]]

    if top_chunks:
        selected_text = top_chunks[0]["text"].lower()
        missing = [kw for kw in keywords if kw not in selected_text]
        if len(missing) > len(keywords) // 2:
            print(f"Chunk missing keywords: {missing} — rejecting")
            top_chunks = []

    context = "\n\n".join([f"Page {c['pages'][0]+1}: {c['text']}" for c in top_chunks])

    prompt = f"""You are a strict document QA system. You only answer from the given context.

ABSOLUTE RULES:
1. If context is empty or says "No context available" → VERDICT must be NOT FOUND, no exceptions.
2. If the question topic does not exist anywhere in the context → VERDICT must be NOT FOUND.
3. NEVER use your own knowledge. NEVER guess. NEVER be helpful outside the context.
4. Only output the format below. Nothing else.

Context:
{context if context else "No context available."}

Question: {question}

Output format:
VERDICT: FOUND or NOT FOUND
EXPLANATION: (only from context, or "Not found in document" if NOT FOUND)
EVIDENCE: (exact text from context, or "None" if NOT FOUND)"""

    print(" Generating answer...")
    result = subprocess.run(
        ["ollama", "run", "llama3.2:3b", prompt],
        capture_output=True, text=True, timeout=90
    )
    answer = result.stdout.strip()

    # If no chunks were found, override answer entirely
    if len(top_chunks) == 0:
        answer = "VERDICT: NOT FOUND\nEXPLANATION: Not found in document\nEVIDENCE: None"

    # ── Debug: see exactly what the LLM returned ──────────────────────────
    print(f"Raw LLM output:\n{repr(answer[:400])}")

    # ── Robust verdict parsing ────────────────────────────────────────────
    # Fix: strip whitespace, case-insensitive match, check NOT FOUND before FOUND
    verdict = "UNKNOWN"
    for line in answer.splitlines():
        line_clean = line.strip()
        if line_clean.upper().startswith("VERDICT:"):
            raw = line_clean.split(":", 1)[1].strip().upper()
            if "NOT FOUND" in raw:
                verdict = "NOT FOUND"
            elif "PARTIAL" in raw:
                verdict = "PARTIAL"
            elif "FOUND" in raw:
                verdict = "FOUND"
            break

    print(f" Parsed verdict: {verdict}")

    # ── Highlighting ──────────────────────────────────────────────────────
    print("Highlighting...")
    doc = fitz.open(PDF_PATH)
    count = 0

    if len(top_chunks) > 0:
        for chunk in top_chunks:
            # Group spans by page — handles chunks spanning multiple pages
            page_spans = {}
            for span_info in chunk.get("spans", []):
                p = span_info["page"]
                if p not in page_spans:
                    page_spans[p] = []
                page_spans[p].append(span_info)

            for page_num, spans in page_spans.items():
                page = doc[page_num]
                for span_info in spans:
                    text = span_info["text"].strip()
                    if not text or len(text) < 3:
                        continue
                    instances = page.search_for(text)
                    if instances:
                        annot = page.add_highlight_annot(instances[0])
                        annot.set_colors(stroke=(1, 0.85, 0.2))
                        annot.set_opacity(0.55)
                        annot.update()
                        count += 1

    output_path = "/content/highlighted.pdf"
    doc.save(output_path)
    doc.close()

    # Save pages as PNG for preview
    doc2 = fitz.open(output_path)
    for page_num in range(len(doc2)):
        page2 = doc2[page_num]
        mat = fitz.Matrix(3, 3)
        pix = page2.get_pixmap(matrix=mat, annots=True)
        pix.save(f"/content/highlighted_page_{page_num+1}.png")
    doc2.close()
    print("Pages saved as PNG")

    print(f"Highlighted {count} spans")
    print("\n" + "="*80)
    print(answer)
    print("="*80)

    pages_hit = sorted({c["pages"][0] + 1 for c in top_chunks}) if top_chunks else []

    return output_path, answer, verdict, pages_hit


print("Functions defined")

In [ ]:

app = Flask(__name__)
CORS(app, origins="*")


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status":  "ok",
        "indexed": chunks is not None,
        "pdf":     PDF_PATH.split("/")[-1] if PDF_PATH else "",
    })


@app.route("/upload", methods=["POST"])
def upload():
    global chunks, embedder, index, PDF_PATH

    if "pdf" not in request.files:
        return jsonify({"error": "No file provided"}), 400

    f = request.files["pdf"]
    if not f.filename.endswith(".pdf"):
        return jsonify({"error": "Only PDF files accepted"}), 400

    # Reset before indexing new PDF
    reset_rag_system()

    # Save uploaded file
    save_path = "/content/" + f.filename
    f.save(save_path)
    PDF_PATH = save_path

    # Load embedder if not already loaded
    if embedder is None:
        print("Loading embedding model...")
        embedder = SentenceTransformer("all-MiniLM-L6-v2")
        print("Embedder ready")

    try:
        # Your original parse_and_chunk
        chunks = parse_and_chunk(PDF_PATH)

        print("Creating embeddings...")
        texts = [c["text"] for c in chunks]
        embeddings = embedder.encode(texts, normalize_embeddings=True)

        print("Building FAISS index...")
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings.astype(np.float32))

        print("Vector database ready!")

        return jsonify({
            "message":  "PDF indexed successfully",
            "filename": f.filename,
            "chunks":   len(chunks),
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/query", methods=["POST"])
def query():
    if chunks is None or index is None:
        return jsonify({"error": "No PDF indexed yet. Upload a PDF first."}), 400

    data     = request.get_json()
    question = (data or {}).get("question", "").strip()
    if not question:
        return jsonify({"error": "No question provided"}), 400

    try:
        #  Your original rag_query — just captures return values
        output_path, answer, verdict, pages_hit = rag_query(question)

        return jsonify({
            "question":     question,
            "answer":       answer,
            "verdict":      verdict,
            "pages":        pages_hit,
            "download_url": "/download",
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/download", methods=["GET"])
def download():
    import os
    if not os.path.exists("/content/highlighted.pdf"):
        return jsonify({"error": "No highlighted PDF yet"}), 404
    return send_file(
        "/content/highlighted.pdf",
        as_attachment=True,
        download_name="highlighted.pdf",
        mimetype="application/pdf",
    )


print("Flask routes defined")



In [ ]:
NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000)

print(f"\n{'='*60}")
print(f"Your API URL:  {public_url}")
print(f"   Copy this into the DocLens frontend → API field")
print(f"{'='*60}\n")

# Pre-load embedder in background so first upload is faster
threading.Thread(target=lambda: SentenceTransformer("all-MiniLM-L6-v2"), daemon=True).start()

# Start Flask — this cell stays running the whole time
app.run(port=5000)